In [ ]:
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm


# ============================================================
# Configuration
# ============================================================

# Paths to the label CSV and the masked RSNA image dataset
CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/boneage-training-dataset.csv"
IMAGE_DIR = "/content/drive/MyDrive/Colab Notebooks/overlayed_RSNA_dataset"

# Paths for saving the best and latest model checkpoints
BEST_CHECKPOINT_PATH = "/content/drive/MyDrive/Colab Notebooks/convnextv2_base_384x512_best.pth"
LATEST_CHECKPOINT_PATH = "/content/drive/MyDrive/Colab Notebooks/convnextv2_base_384x512_latest.pth"

# Directory for plots, logs, predictions, split files and config output
PLOT_DIR = "/content/drive/MyDrive/Colab Notebooks/training_plots_convnextv2_base_384x512"

# Resume settings for continuing interrupted training runs
RESUME_TRAINING = True
RESUME_FROM_BEST_IF_NO_LATEST = True

# ConvNeXtV2 backbone from timm
MODEL_NAME = "convnextv2_base.fcmae_ft_in22k_in1k_384"

# Input image size: width x height = 384 x 512
IMAGE_HEIGHT = 512
IMAGE_WIDTH = 384

# If False, ImageNet-pretrained weights are used
NO_PRETRAINED = False

# Split and reproducibility settings
VAL_SIZE = 0.15
SEED = 42

# Batch settings
# Effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
NUM_WORKERS = 2

# Training schedule and early stopping
EPOCHS = 100
WARMUP_EPOCHS = 5
PATIENCE = 15

# Separate learning rates for backbone and regression head
BACKBONE_LR = 1e-4
HEAD_LR = 3e-4
WEIGHT_DECAY = 0.05

# Regularization and regression head settings
DROP_PATH = 0.1
HEAD_DROPOUT = 0.2
HIDDEN_DIM = 512

# SmoothL1Loss beta and gradient clipping threshold
SMOOTH_L1_BETA = 6.0
MAX_GRAD_NORM = 1.0

# Mixed precision training for GPU acceleration
USE_AMP = True

# Supported image formats for indexing the image directory
SUPPORTED_IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int) -> None:
    """
    Set random seeds for reproducibility.
    This affects Python, NumPy and PyTorch randomness.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ============================================================
# Image indexing / CSV filtering
# ============================================================

def normalize_id(value) -> str:
    """
    Normalize image IDs so that CSV IDs and image filenames can be matched reliably.
    Example: 1377.0 becomes "1377".
    """
    if pd.isna(value):
        return ""

    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    return str(value).strip()


def build_image_index(image_dir):
    """
    Build a dictionary that maps image IDs to actual image file paths.
    This avoids repeatedly searching the image directory during training.
    """
    image_dir = Path(image_dir)

    if not image_dir.exists():
        raise FileNotFoundError(f"IMAGE_DIR does not exist: {image_dir}")

    image_index = {}

    for ext in SUPPORTED_IMAGE_EXTENSIONS:
        for path in image_dir.glob(f"*{ext}"):
            image_index[path.stem] = path

    return image_index


def filter_dataframe_to_existing_images(df, image_index, plot_dir):
    """
    Keep only CSV rows for which an image file exists.
    Missing IDs are saved to missing_images.csv for traceability.
    """
    df = df.copy()
    df["id"] = df["id"].apply(normalize_id)

    exists_mask = df["id"].isin(image_index.keys())
    missing_df = df.loc[~exists_mask].copy()
    filtered_df = df.loc[exists_mask].copy()

    plot_dir = Path(plot_dir)
    plot_dir.mkdir(parents=True, exist_ok=True)

    if len(missing_df) > 0:
        missing_path = plot_dir / "missing_images.csv"
        missing_df.to_csv(missing_path, index=False)

        print(f"Warning: {len(missing_df)} rows were removed because no image file was found.")
        print(f"Missing image IDs saved to: {missing_path}")
        print("First missing IDs:", missing_df["id"].head(20).tolist())
    else:
        print("All CSV rows have matching image files.")

    print(f"Samples after image-file filtering: {len(filtered_df)}")

    return filtered_df


# ============================================================
# Dataset
# ============================================================

class BoneAgeDataset(Dataset):
    """
    Custom PyTorch Dataset for bone age prediction.

    Each sample returns:
    - preprocessed image tensor
    - gender metadata tensor
    - bone age target
    - image ID
    """
    def __init__(self, dataframe, image_index, image_height, image_width):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_index = image_index

        # Resize to the configured rectangular input size and normalize
        # with ImageNet statistics because the backbone is pretrained.
        self.transform = transforms.Compose([
            transforms.Resize((image_height, image_width)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Match CSV ID to image path
        image_id = normalize_id(row["id"])
        image_path = self.image_index[image_id]

        # Convert grayscale masked X-ray to RGB because pretrained CNNs expect 3 channels
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        # Bone age is the regression target in months
        boneage = torch.tensor(float(row["boneage"]), dtype=torch.float32)

        # Convert male column to numeric metadata:
        # male = 1.0, female = 0.0
        male_value = row["male"]
        if isinstance(male_value, str):
            male_value = male_value.lower() == "true"

        male = torch.tensor([float(male_value)], dtype=torch.float32)

        return {
            "image": image,
            "male": male,
            "target": boneage,
            "id": image_id,
        }


# ============================================================
# Model
# ============================================================

class BoneAgeConvNeXtV2(nn.Module):
    """
    ConvNeXtV2 regression model with gender metadata.

    The image backbone extracts visual features.
    The gender feature is concatenated to the image features.
    The regression head predicts bone age in months.
    """
    def __init__(
        self,
        model_name: str,
        pretrained: bool = True,
        drop_path_rate: float = 0.1,
        head_dropout: float = 0.2,
        hidden_dim: int = 512,
    ):
        super().__init__()

        # Create ConvNeXtV2 backbone without classification head
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
            drop_path_rate=drop_path_rate,
        )

        feature_dim = self.backbone.num_features

        # Regression head receives image features plus one gender feature
        self.regression_head = nn.Sequential(
            nn.Linear(feature_dim + 1, hidden_dim),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, image, male):
        # Extract image embedding from ConvNeXtV2
        features = self.backbone(image)

        # Concatenate gender metadata to image features
        features = torch.cat([features, male], dim=1)

        # Predict one continuous bone age value
        prediction = self.regression_head(features).squeeze(1)
        return prediction


# ============================================================
# Split
# ============================================================

def create_stratified_split(df, val_size, seed):
    """
    Create a train/validation split stratified by age bins and gender.

    This helps keep the validation set representative across age ranges
    and male/female distribution.
    """
    df = df.copy()

    # Divide bone age into quantile-based bins
    df["age_bin"] = pd.qcut(
        df["boneage"],
        q=10,
        duplicates="drop",
        labels=False,
    )

    # Combine age bin and gender for stratification
    df["male_str"] = df["male"].astype(str)
    df["stratify_col"] = df["age_bin"].astype(str) + "_" + df["male_str"]

    stratify_col = df["stratify_col"]

    # Fallback if combined age+gender strata are too small
    if stratify_col.value_counts().min() < 2:
        print(
            "Warning: Some age+gender strata contain fewer than 2 samples. "
            "Falling back to stratification by age_bin only."
        )
        stratify_col = df["age_bin"]

    # Final fallback to random split if age bins are still too small
    if pd.Series(stratify_col).value_counts().min() < 2:
        print(
            "Warning: Some age bins contain fewer than 2 samples. "
            "Falling back to random split."
        )
        stratify_col = None

    train_df, val_df = train_test_split(
        df,
        test_size=val_size,
        random_state=seed,
        shuffle=True,
        stratify=stratify_col,
    )

    # Remove helper columns before creating datasets
    train_df = train_df.drop(columns=["age_bin", "male_str", "stratify_col"])
    val_df = val_df.drop(columns=["age_bin", "male_str", "stratify_col"])

    return train_df, val_df


# ============================================================
# Optimizer
# ============================================================

def build_optimizer(model, backbone_lr, head_lr, weight_decay):
    """
    Build AdamW optimizer with separate parameter groups:
    - backbone parameters with weight decay
    - bias/norm-like parameters without weight decay
    - regression head with higher learning rate
    """
    decay_params = []
    no_decay_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if name.startswith("regression_head"):
            head_params.append(param)
        elif param.ndim < 2 or name.endswith(".bias"):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": decay_params,
                "lr": backbone_lr,
                "weight_decay": weight_decay,
            },
            {
                "params": no_decay_params,
                "lr": backbone_lr,
                "weight_decay": 0.0,
            },
            {
                "params": head_params,
                "lr": head_lr,
                "weight_decay": weight_decay,
            },
        ]
    )

    return optimizer


# ============================================================
# Scheduler
# ============================================================

def build_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs):
    """
    Learning-rate schedule:
    - linear warmup for the first warmup_epochs
    - cosine decay afterwards
    """
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))

        progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ============================================================
# Metrics
# ============================================================

def compute_mae(preds, targets):
    """
    Compute mean absolute error in months.
    """
    preds = np.asarray(preds)
    targets = np.asarray(targets)
    return float(np.mean(np.abs(preds - targets)))


def compute_rmse(preds, targets):
    """
    Compute root mean squared error in months.
    """
    preds = np.asarray(preds)
    targets = np.asarray(targets)
    return float(np.sqrt(np.mean((preds - targets) ** 2)))


def group_metrics(preds, targets, males):
    """
    Compute MAE summaries by age group and by gender.
    These diagnostics show whether the model performs worse for specific subgroups.
    """
    df = pd.DataFrame({
        "pred": preds,
        "target": targets,
        "male": males,
    })

    df["abs_error"] = (df["pred"] - df["target"]).abs()

    age_bins = [0, 24, 48, 72, 96, 120, 144, 168, 192, 240]
    df["age_group"] = pd.cut(
        df["target"],
        bins=age_bins,
        right=False,
        include_lowest=True,
    )

    age_metrics = (
        df.groupby("age_group", observed=False)["abs_error"]
        .agg(["count", "mean", "median"])
        .reset_index()
    )

    gender_metrics = (
        df.groupby("male")["abs_error"]
        .agg(["count", "mean", "median"])
        .reset_index()
    )

    return age_metrics, gender_metrics


# ============================================================
# Train / Validate
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    scaler,
    use_amp,
    grad_accum_steps,
    max_grad_norm,
):
    """
    Train the model for one epoch.

    Uses optional mixed precision, gradient accumulation and gradient clipping.
    Returns average training loss and training MAE.
    """
    model.train()

    running_loss = 0.0
    all_preds = []
    all_targets = []

    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for step, batch in enumerate(progress_bar):
        images = batch["image"].to(device, non_blocking=True)
        males = batch["male"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)

        # Forward pass under mixed precision if enabled
        with torch.amp.autocast(
            device_type="cuda",
            enabled=use_amp and device.type == "cuda",
        ):
            preds = model(images, males)
            loss = criterion(preds, targets)

            # Divide loss so accumulated gradients match the effective batch size
            loss_for_backward = loss / grad_accum_steps

        # Backward pass with or without AMP scaler
        if scaler is not None:
            scaler.scale(loss_for_backward).backward()
        else:
            loss_for_backward.backward()

        should_step = (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader)

        if should_step:
            if scaler is not None:
                scaler.unscale_(optimizer)

            # Clip gradients to stabilize training
            if max_grad_norm is not None and max_grad_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            # Optimizer step with or without AMP scaler
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * images.size(0)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(targets.detach().cpu().numpy().tolist())

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "mae": f"{compute_mae(all_preds, all_targets):.2f}",
        })

    epoch_loss = running_loss / len(loader.dataset)
    epoch_mae = compute_mae(all_preds, all_targets)

    return epoch_loss, epoch_mae


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, use_amp):
    """
    Evaluate the model on the validation set.

    Returns:
    - validation loss
    - validation MAE
    - validation RMSE
    - dataframe with predictions, targets, gender and errors
    """
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_targets = []
    all_males = []
    all_ids = []

    progress_bar = tqdm(loader, desc="Validation", leave=False)

    for batch in progress_bar:
        images = batch["image"].to(device, non_blocking=True)
        males = batch["male"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=use_amp and device.type == "cuda",
        ):
            preds = model(images, males)
            loss = criterion(preds, targets)

        running_loss += loss.item() * images.size(0)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(targets.detach().cpu().numpy().tolist())
        all_males.extend(males.detach().cpu().numpy().reshape(-1).tolist())
        all_ids.extend(batch["id"])

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "mae": f"{compute_mae(all_preds, all_targets):.2f}",
        })

    val_loss = running_loss / len(loader.dataset)
    val_mae = compute_mae(all_preds, all_targets)
    val_rmse = compute_rmse(all_preds, all_targets)

    predictions_df = pd.DataFrame({
        "id": all_ids,
        "target": all_targets,
        "prediction": all_preds,
        "male": all_males,
        "abs_error": np.abs(np.asarray(all_preds) - np.asarray(all_targets)),
    })

    return val_loss, val_mae, val_rmse, predictions_df


# ============================================================
# Checkpointing
# ============================================================

def save_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    scheduler,
    best_val_mae,
    best_epoch,
    patience_counter,
    history,
    config,
):
    """
    Save complete training state so training can be resumed later.
    """
    checkpoint = {
        "epoch": epoch,
        "model_name": MODEL_NAME,
        "image_height": IMAGE_HEIGHT,
        "image_width": IMAGE_WIDTH,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_mae": best_val_mae,
        "best_epoch": best_epoch,
        "patience_counter": patience_counter,
        "history": history,
        "config": config,
    }

    torch.save(checkpoint, checkpoint_path)


def load_history_from_csv_if_available():
    """
    Load history.csv if checkpoint history is missing.
    This keeps plots continuous after resuming from older checkpoints.
    """
    history_path = Path(PLOT_DIR) / "history.csv"

    if history_path.exists():
        history_df = pd.read_csv(history_path)
        history = history_df.to_dict("records")
        print(f"Loaded history from: {history_path}")
        print(f"History entries loaded: {len(history)}")
        return history

    print("No history found in checkpoint or history.csv. History will continue from resume point.")
    return []


def resume_from_checkpoint_if_available(
    model,
    optimizer,
    scheduler,
    device,
):
    """
    Resume training from the latest checkpoint if available.
    If no latest checkpoint exists, optionally resume from the best checkpoint.
    """
    start_epoch = 1
    history = []
    best_val_mae = float("inf")
    best_epoch = -1
    patience_counter = 0

    latest_path = Path(LATEST_CHECKPOINT_PATH)
    best_path = Path(BEST_CHECKPOINT_PATH)

    checkpoint_path = None

    if RESUME_TRAINING and latest_path.exists():
        checkpoint_path = latest_path
        print(f"Resuming from latest checkpoint: {checkpoint_path}")
    elif RESUME_TRAINING and RESUME_FROM_BEST_IF_NO_LATEST and best_path.exists():
        checkpoint_path = best_path
        print(f"Latest checkpoint not found. Resuming from best checkpoint: {checkpoint_path}")
    else:
        print("No checkpoint loaded. Starting from scratch.")
        return start_epoch, history, best_val_mae, best_epoch, patience_counter

    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])

    if "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    else:
        print("Warning: optimizer_state_dict not found. Optimizer starts fresh.")

    if "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    else:
        print("Warning: scheduler_state_dict not found. Scheduler starts fresh.")

    checkpoint_epoch = int(checkpoint.get("epoch", 0))
    start_epoch = checkpoint_epoch + 1

    best_val_mae = float(checkpoint.get("best_val_mae", float("inf")))
    best_epoch = int(checkpoint.get("best_epoch", checkpoint_epoch))
    patience_counter = int(checkpoint.get("patience_counter", 0))

    history = checkpoint.get("history", [])

    if len(history) == 0:
        history = load_history_from_csv_if_available()

    print(f"Checkpoint epoch: {checkpoint_epoch}")
    print(f"Next epoch: {start_epoch}")
    print(f"Best validation MAE so far: {best_val_mae:.4f} months")
    print(f"Best epoch so far: {best_epoch}")
    print(f"Patience counter: {patience_counter}")

    return start_epoch, history, best_val_mae, best_epoch, patience_counter


# ============================================================
# Plotting
# ============================================================

def save_plots(history_df, predictions_df, plot_dir):
    """
    Save training diagnostic plots:
    - loss curves
    - MAE curves
    - true vs predicted validation scatter plot
    - MAE by age group
    - residual histogram
    """
    plot_dir = Path(plot_dir)
    plot_dir.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], label="Train loss")
    axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], label="Val loss")
    axes[0, 0].set_title("Loss curves")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("SmoothL1 loss")
    axes[0, 0].legend()

    axes[0, 1].plot(history_df["epoch"], history_df["train_mae"], label="Train MAE")
    axes[0, 1].plot(history_df["epoch"], history_df["val_mae"], label="Val MAE")
    axes[0, 1].set_title("MAE curves")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("MAE in months")
    axes[0, 1].legend()

    # Scatter plot of target vs prediction
    # Ideal predictions lie on the diagonal line
    axes[1, 0].scatter(
        predictions_df["target"],
        predictions_df["prediction"],
        alpha=0.5,
        s=12,
    )

    min_age = min(predictions_df["target"].min(), predictions_df["prediction"].min())
    max_age = max(predictions_df["target"].max(), predictions_df["prediction"].max())

    axes[1, 0].plot([min_age, max_age], [min_age, max_age], linestyle="--")
    axes[1, 0].set_title("Validation predictions")
    axes[1, 0].set_xlabel("True bone age in months")
    axes[1, 0].set_ylabel("Predicted bone age in months")

    # Group validation errors by bone age range
    age_bins = [0, 24, 48, 72, 96, 120, 144, 168, 192, 240]
    predictions_df = predictions_df.copy()
    predictions_df["age_group"] = pd.cut(
        predictions_df["target"],
        bins=age_bins,
        right=False,
        include_lowest=True,
    )

    group_mae = (
        predictions_df.groupby("age_group", observed=False)["abs_error"]
        .mean()
        .reset_index()
    )

    axes[1, 1].bar(
        group_mae["age_group"].astype(str),
        group_mae["abs_error"],
    )
    axes[1, 1].set_title("Validation MAE by age group")
    axes[1, 1].set_xlabel("Age group in months")
    axes[1, 1].set_ylabel("MAE in months")
    axes[1, 1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(plot_dir / "training_diagnostics.png", dpi=200)
    plt.close()

    # Residuals show the signed prediction error distribution
    residuals = predictions_df["prediction"] - predictions_df["target"]

    plt.figure(figsize=(8, 5))
    plt.hist(residuals, bins=40)
    plt.title("Validation residuals")
    plt.xlabel("Prediction error in months")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(plot_dir / "validation_residuals.png", dpi=200)
    plt.close()


# ============================================================
# Main
# ============================================================

def main():
    """
    Full training pipeline:
    1. Set seed and prepare output folders
    2. Index available images
    3. Load and filter CSV labels
    4. Create stratified train/validation split
    5. Build datasets and dataloaders
    6. Initialize ConvNeXtV2 model, optimizer, scheduler and AMP scaler
    7. Resume from checkpoint if available
    8. Train with early stopping
    9. Save checkpoints, predictions, plots and subgroup metrics
    """
    set_seed(SEED)

    plot_dir = Path(PLOT_DIR)
    plot_dir.mkdir(parents=True, exist_ok=True)

    best_checkpoint_path = Path(BEST_CHECKPOINT_PATH)
    latest_checkpoint_path = Path(LATEST_CHECKPOINT_PATH)

    best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    latest_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Build image ID to file path mapping
    image_index = build_image_index(IMAGE_DIR)
    print(f"Found image files: {len(image_index)}")

    # Load label CSV
    df = pd.read_csv(CSV_PATH)

    required_columns = {"id", "boneage", "male"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(f"CSV is missing required columns: {missing_columns}")

    df["id"] = df["id"].apply(normalize_id)

    print(f"Total CSV samples: {len(df)}")
    print(f"Bone age range: {df['boneage'].min()} to {df['boneage'].max()} months")
    print(f"Male distribution:\n{df['male'].value_counts()}")

    # Remove rows without matching image files
    df = filter_dataframe_to_existing_images(
        df=df,
        image_index=image_index,
        plot_dir=plot_dir,
    )

    # Create train/validation split
    train_df, val_df = create_stratified_split(
        df=df,
        val_size=VAL_SIZE,
        seed=SEED,
    )

    print(f"Train samples: {len(train_df)}")
    print(f"Validation samples: {len(val_df)}")

    # Save exact split for reproducibility
    train_df.to_csv(plot_dir / "train_split.csv", index=False)
    val_df.to_csv(plot_dir / "val_split.csv", index=False)

    train_dataset = BoneAgeDataset(
        dataframe=train_df,
        image_index=image_index,
        image_height=IMAGE_HEIGHT,
        image_width=IMAGE_WIDTH,
    )

    val_dataset = BoneAgeDataset(
        dataframe=val_df,
        image_index=image_index,
        image_height=IMAGE_HEIGHT,
        image_width=IMAGE_WIDTH,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )

    # Initialize model
    model = BoneAgeConvNeXtV2(
        model_name=MODEL_NAME,
        pretrained=not NO_PRETRAINED,
        drop_path_rate=DROP_PATH,
        head_dropout=HEAD_DROPOUT,
        hidden_dim=HIDDEN_DIM,
    )

    model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Model: {MODEL_NAME}")
    print(f"Image size: {IMAGE_WIDTH}x{IMAGE_HEIGHT} width x height")
    print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # SmoothL1Loss is less sensitive to large errors than MSE
    criterion = nn.SmoothL1Loss(beta=SMOOTH_L1_BETA)

    optimizer = build_optimizer(
        model=model,
        backbone_lr=BACKBONE_LR,
        head_lr=HEAD_LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = build_warmup_cosine_scheduler(
        optimizer=optimizer,
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=EPOCHS,
    )

    # Enable AMP only when CUDA is available
    use_amp = USE_AMP and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    # Save all relevant experiment settings for reproducibility
    config = {
        "csv_path": CSV_PATH,
        "image_dir": IMAGE_DIR,
        "best_checkpoint_path": BEST_CHECKPOINT_PATH,
        "latest_checkpoint_path": LATEST_CHECKPOINT_PATH,
        "plot_dir": PLOT_DIR,
        "resume_training": RESUME_TRAINING,
        "resume_from_best_if_no_latest": RESUME_FROM_BEST_IF_NO_LATEST,
        "model_name": MODEL_NAME,
        "image_height": IMAGE_HEIGHT,
        "image_width": IMAGE_WIDTH,
        "no_pretrained": NO_PRETRAINED,
        "val_size": VAL_SIZE,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUM_STEPS,
        "num_workers": NUM_WORKERS,
        "epochs": EPOCHS,
        "warmup_epochs": WARMUP_EPOCHS,
        "patience": PATIENCE,
        "backbone_lr": BACKBONE_LR,
        "head_lr": HEAD_LR,
        "weight_decay": WEIGHT_DECAY,
        "drop_path": DROP_PATH,
        "head_dropout": HEAD_DROPOUT,
        "hidden_dim": HIDDEN_DIM,
        "smooth_l1_beta": SMOOTH_L1_BETA,
        "max_grad_norm": MAX_GRAD_NORM,
        "use_amp": USE_AMP,
        "device": str(device),
        "total_params": total_params,
        "trainable_params": trainable_params,
        "found_image_files": len(image_index),
        "training_samples_after_filtering": len(df),
    }

    with open(plot_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    # Resume previous training if a checkpoint is available
    start_epoch, history, best_val_mae, best_epoch, patience_counter = resume_from_checkpoint_if_available(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
    )

    if start_epoch > EPOCHS:
        print(f"Checkpoint already reached epoch {start_epoch - 1}. EPOCHS is set to {EPOCHS}.")
        print("Increase EPOCHS if you want to continue training further.")
        return

    predictions_df = None

    for epoch in range(start_epoch, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")

        train_loss, train_mae = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            use_amp=use_amp,
            grad_accum_steps=GRAD_ACCUM_STEPS,
            max_grad_norm=MAX_GRAD_NORM,
        )

        val_loss, val_mae, val_rmse, predictions_df = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            use_amp=use_amp,
        )

        scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]

        epoch_log = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_mae": train_mae,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse,
            "lr": current_lr,
        }

        history.append(epoch_log)

        print(
            f"Train loss: {train_loss:.4f} | "
            f"Train MAE: {train_mae:.2f} months | "
            f"Val loss: {val_loss:.4f} | "
            f"Val MAE: {val_mae:.2f} months | "
            f"Val RMSE: {val_rmse:.2f} months | "
            f"LR: {current_lr:.2e}"
        )

        # Save history and latest validation predictions after every epoch
        history_df = pd.DataFrame(history)
        history_df.to_csv(plot_dir / "history.csv", index=False)

        predictions_df.to_csv(plot_dir / "latest_val_predictions.csv", index=False)

        # Save best checkpoint based on validation MAE
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_epoch = epoch
            patience_counter = 0

            save_checkpoint(
                checkpoint_path=best_checkpoint_path,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                best_val_mae=best_val_mae,
                best_epoch=best_epoch,
                patience_counter=patience_counter,
                history=history,
                config=config,
            )

            predictions_df.to_csv(plot_dir / "best_val_predictions.csv", index=False)

            print(f"Saved new best model to: {best_checkpoint_path}")
            print(f"Best Val MAE: {best_val_mae:.2f} months")
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")

        # Always save latest checkpoint for safe resume
        save_checkpoint(
            checkpoint_path=latest_checkpoint_path,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_val_mae=best_val_mae,
            best_epoch=best_epoch,
            patience_counter=patience_counter,
            history=history,
            config=config,
        )

        print(f"Saved latest checkpoint to: {latest_checkpoint_path}")

        # Stop when validation MAE has not improved for PATIENCE epochs
        if patience_counter >= PATIENCE:
            print(
                f"Early stopping triggered. "
                f"Best epoch: {best_epoch}, Best Val MAE: {best_val_mae:.2f} months"
            )
            break

    history_df = pd.DataFrame(history)
    best_predictions_path = plot_dir / "best_val_predictions.csv"

    # Use best predictions for final plots if available
    if best_predictions_path.exists():
        best_predictions_df = pd.read_csv(best_predictions_path)
    elif predictions_df is not None:
        best_predictions_df = predictions_df
    else:
        latest_predictions_path = plot_dir / "latest_val_predictions.csv"
        if latest_predictions_path.exists():
            best_predictions_df = pd.read_csv(latest_predictions_path)
        else:
            print("No prediction file found. Skipping final plots.")
            return

    save_plots(
        history_df=history_df,
        predictions_df=best_predictions_df,
        plot_dir=plot_dir,
    )

    # Save subgroup diagnostics
    age_metrics, gender_metrics = group_metrics(
        preds=best_predictions_df["prediction"].values,
        targets=best_predictions_df["target"].values,
        males=best_predictions_df["male"].values,
    )

    age_metrics.to_csv(plot_dir / "mae_by_age_group.csv", index=False)
    gender_metrics.to_csv(plot_dir / "mae_by_gender.csv", index=False)

    print("\nTraining finished.")
    print(f"Best epoch: {best_epoch}")
    print(f"Best validation MAE: {best_val_mae:.2f} months")
    print(f"Best model saved to: {best_checkpoint_path}")
    print(f"Latest checkpoint saved to: {latest_checkpoint_path}")
    print(f"Plots and logs saved to: {plot_dir}")

    print("\nMAE by age group:")
    print(age_metrics)

    print("\nMAE by gender:")
    print(gender_metrics)


if __name__ == "__main__":
    main()

Using device: cuda


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/boneage-training-dataset.csv'

In [ ]:
csv_file_path = Path(CSV_PATH)
if not csv_file_path.exists():
    raise FileNotFoundError(
        f"CSV file not found at {CSV_PATH}. "
        "Please ensure the file is correctly placed in your Google Drive and mounted."
    )

df = pd.read_csv(CSV_PATH)

required_columns = {"id", "boneage", "male"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

print(f"Total samples: {len(df)}")
print(f"Bone age range: {df['boneage'].min()} to {df['boneage'].max()} months")
print(f"Male distribution:\n{df['male'].value_counts()}")

FileNotFoundError: CSV file not found at /content/drive/MyDrive/Colab Notebooks/boneage-training-dataset.csv. Please ensure the file is correctly placed in your Google Drive and mounted.

In [ ]:
import os

print(os.listdir('/content/drive/MyDrive/Colab Notebooks'))
print(os.cpu_count())

['boneage-training-dataset.csv', 'overlayed_RSNA_dataset', 'training_plots', 'resnet50_best.pth', 'resnet50_train.ipynb']
2


In [ ]:
!nvidia-smi

Sat Apr 25 23:30:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             28W /   70W |   13211MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----